# Uncertainty-aware link anomaly detection in CTDGs (GraphMixer)

This notebook compares two runs that use the same code, data, seeds and hyper-parameters:

* **Baseline**: GraphMixer exactly as in *Learning-Based Link Anomaly Detection in Continuous-Time Dynamic Graphs* (TMLR 2024).
* **Uncertainty**: the same model with the prediction head replaced by `models/uncertainty.py::UncertaintyLinkPredictor`
  (+61 trainable parameters for GraphMixer, i.e. hidden_dim + 1; identical initial weights), which models
  * **aleatoric** uncertainty with a heteroscedastic logit head trained by MC-integrating the likelihood,
  * **epistemic** uncertainty with MC dropout in the head (the backbone runs once per edge),
  * **support** uncertainty `W / (W + n)` from the number `n` of earlier observations of the node pair,

  and scores each edge with a lower confidence bound `p(e) - Σ λ_i u_i(e)`, where the `λ_i ≥ 0` are selected on the **validation** set only.

The only difference between the two runs is the `uncertainty:` section of the config. Without it, the original pipeline from the paper runs unchanged.

**Before running:** push this repository (including `models/uncertainty.py` and `download_datasets.py`) to your GitHub and set `REPO_URL` below, or attach the repository as a Kaggle dataset. Use a **GPU** accelerator and turn **Internet on**.

In [ ]:
# ================================ SETTINGS ================================
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/CTDG-link-anomaly-detection"  # your fork with the uncertainty module
REPO_BRANCH = "main"

# Datasets from Zenodo (Poursafaei et al., 2022): wikipedia, reddit, mooc, enron, uci, lastfm, SocialEvo; or "synthetic".
DATASETS = ["wikipedia"]
# Anomaly types from the paper. "contextual" and "temporal-contextual" need edge features (not on enron / uci).
ANOM_TYPES = ["structural-contextual", "temporal", "temporal-structural-contextual"]

# Paper settings: 50 epochs, patience 20, 3 runs x 3 anomaly sets, learning-rate grid [1e-3 ... 1e-6].
# Defaults below are reduced so that the whole notebook fits into one Kaggle session.
NUM_EPOCHS = 20
PATIENCE = 20
LEARNING_RATES = [1e-4]
NUM_RUNS = 1
NUM_ANOM_SETS = 1

RUN_BASELINE = True     # reproduces the paper's GraphMixer
RUN_UNCERTAINTY = True  # GraphMixer + uncertainty module

UNCERTAINTY = dict(
    components=["aleatoric", "epistemic", "support"],  # any subset
    mc_dropout=0.1,
    num_mc_samples=20,
    num_noise_samples=10,
    init_logit_std=0.1,
    support_prior_weight=2.0,
    lambda_grid=[0.0, 0.01, 0.025, 0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2],
)

# GraphMixer hyper-parameters from experiments/experiment.yaml (the paper's configuration).
GRAPHMIXER = dict(
    node_feat_dim=60,
    edge_feat_dim=60,
    num_neighbors=20,
    sample_neighbor_strategy="recent",
    time_feat_dim=60,
    time_gap=2000,
    num_layers=2,
    dropout=0.4,
)

WORK_DIR = "/kaggle/working"
REPO_DIR = f"{WORK_DIR}/CTDG-link-anomaly-detection"
TGB_DIR = f"{WORK_DIR}/TGB-link-anomaly-detection"
DATA_DIR = f"{WORK_DIR}/DATASETS"
OUT_DIR = f"{WORK_DIR}/EXPERIMENTS"

## 1. Get the code

In [ ]:
import glob, os, shutil

if not os.path.isdir(REPO_DIR):
    # Prefer a copy attached as a Kaggle dataset, otherwise clone from GitHub.
    attached = glob.glob("/kaggle/input/**/train_tgb_linkanomdet.py", recursive=True)
    if attached:
        shutil.copytree(os.path.dirname(attached[0]), REPO_DIR)
    else:
        assert "<YOUR_GITHUB_USERNAME>" not in REPO_URL, "Set REPO_URL in the settings cell (or attach the repo as a dataset)."
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
assert os.path.exists("models/uncertainty.py") and os.path.exists("download_datasets.py"), \
    "The repository does not contain the uncertainty module - push models/uncertainty.py and download_datasets.py."
!git log --oneline -3 2>/dev/null || true

## 2. Install dependencies
Kaggle already has PyTorch, NumPy, pandas and scikit-learn.

In [ ]:
!pip install -q torch_geometric omegaconf "mlflow-skinny>=2.10,<3" clint
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Download datasets and generate anomalies
`download_datasets.py` clones [TGB-link-anomaly-detection](https://github.com/timpostuvan/TGB-link-anomaly-detection), downloads the datasets without the interactive prompt, and injects 5% typed anomalies into the validation and test splits (same generator and seed as in the paper).
TGB-link-anomaly-detection has no `__init__.py` files, so it is put on `PYTHONPATH` instead of being pip-installed.

In [ ]:
os.environ["PYTHONPATH"] = TGB_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
DATASETS_ARG = " ".join(DATASETS)
ANOM_TYPES_ARG = " ".join(ANOM_TYPES)
!python download_datasets.py --dataset_root {DATA_DIR} --tgb_dir {TGB_DIR} --datasets {DATASETS_ARG} --anom_types {ANOM_TYPES_ARG} --num_anom_sets {NUM_ANOM_SETS}
!python -c "import tgb.linkanomdet.dataset_pyg as m; print('tgb found at', m.__file__)"

## 4. Write the experiment configs
The two configs are identical except for the `uncertainty` section.

In [ ]:
from omegaconf import OmegaConf

def make_config(experiment_name, uncertainty=None):
    config = {
        "general": {
            "show_progress_bar": False,
            "experiment_name": experiment_name,
            "num_runs": NUM_RUNS,
            "num_anom_sets": NUM_ANOM_SETS,
            "mlflow_tracking_uri": f"{OUT_DIR}/mlflow",
            "output_root": OUT_DIR,
        },
        "data": {
            "dataset_root": DATA_DIR,
            "dataset_name": DATASETS,
            "anom_type": ANOM_TYPES,
            "val_ratio": 0.15,
            "test_ratio": 0.15,
        },
        "models": {"GraphMixer": GRAPHMIXER},
        "training": {
            "learning_rate": LEARNING_RATES,
            "num_epochs": NUM_EPOCHS,
            "batch_size": 200,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "patience": PATIENCE,
            "test_interval_epochs": 10,
            "seed": 2023,
        },
    }
    if uncertainty is not None:
        config["uncertainty"] = {"enabled": True, **uncertainty}
    return OmegaConf.create(config)

BASELINE_CFG = f"{WORK_DIR}/config_baseline.yaml"
UNCERTAINTY_CFG = f"{WORK_DIR}/config_uncertainty.yaml"
OmegaConf.save(make_config("Baseline"), BASELINE_CFG)
OmegaConf.save(make_config("Uncertainty", UNCERTAINTY), UNCERTAINTY_CFG)
!diff {BASELINE_CFG} {UNCERTAINTY_CFG} || true

## 5. Baseline: GraphMixer from the paper

In [ ]:
if RUN_BASELINE:
    !python train_tgb_linkanomdet.py --config_path={BASELINE_CFG}

## 6. GraphMixer + uncertainty module

In [ ]:
if RUN_UNCERTAINTY:
    !python train_tgb_linkanomdet.py --config_path={UNCERTAINTY_CFG}

## 7. Results
For every method, dataset and anomaly type, the learning rate is selected by validation AUC (as in the paper), and test metrics are averaged over runs and anomaly sets.

* **Baseline**: the paper's GraphMixer.
* **Unc. (mean only)**: the uncertainty-trained model scored by its probability alone, which isolates the effect of the heteroscedastic training.
* **Unc. (full)**: the uncertainty-aware score `p - Σ λ_i u_i`.

In [ ]:
import json
import numpy as np
import pandas as pd

def load_results(experiment_name):
    rows = []
    for path in glob.glob(f"{OUT_DIR}/saved_results/{experiment_name}/*/*/*/*.json"):
        anom_dir, dataset, model = path.split(os.sep)[-4:-1]
        with open(path) as file:
            result = json.load(file)
        lr = float(os.path.basename(path).split("_lr_")[1].split("_")[0])
        rows.append({
            "method": experiment_name,
            "dataset": dataset,
            "anomaly": anom_dir[: len(anom_dir) // 2],  # "<val type>-<test type>" with equal types
            "model": model,
            "lr": lr,
            "val auc": max(result["validation auc"]),
            **{k[len("test "):]: v for k, v in result.items() if k.startswith("test ") and k != "test_time"},
        })
    return pd.DataFrame(rows)

def select_lr(df):
    # Learning rate with the best mean validation AUC per (method, dataset, anomaly).
    keys = ["method", "dataset", "anomaly"]
    best_lr = df.groupby(keys + ["lr"])["val auc"].mean().reset_index().sort_values("val auc").groupby(keys).tail(1)
    return df.merge(best_lr[keys + ["lr"]], on=keys + ["lr"])

results = pd.concat([load_results(m) for m in ["Baseline", "Uncertainty"]], ignore_index=True)
assert len(results), "No results found - did the training cells run?"
results = select_lr(results)

def mean_std(x):
    x = 100 * np.asarray(x, dtype=float)
    return f"{x.mean():.2f} ± {x.std():.2f}" if len(x) > 1 else f"{x.mean():.2f}"

summary = []
for (dataset, anomaly), group in results.groupby(["dataset", "anomaly"]):
    base = group[group.method == "Baseline"]
    unc = group[group.method == "Uncertainty"]
    row = {"dataset": dataset, "anomaly": anomaly}
    for metric in ["auc", "ap", "recall@k"]:
        row[f"Baseline {metric}"] = mean_std(base[metric]) if len(base) else "-"
        if len(unc):
            row[f"Unc. (mean only) {metric}"] = mean_std(unc[f"{metric}_mean_only"])
            row[f"Unc. (full) {metric}"] = mean_std(unc[metric])
        if len(base) and len(unc):
            row[f"Δ {metric}"] = f"{100 * (unc[metric].mean() - base[metric].mean()):+.2f}"
    summary.append(row)
summary = pd.DataFrame(summary).set_index(["dataset", "anomaly"])

pd.set_option("display.max_columns", None, "display.width", 250)
print("Test AUC (x100)")
display(summary[[c for c in summary.columns if c.endswith(" auc")]])
print("Test AP / Recall@k (x100)")
display(summary[[c for c in summary.columns if not c.endswith(" auc")]])
summary.to_csv(f"{WORK_DIR}/summary.csv")

### What the uncertainty module learned
`lambda_*` are the weights selected on validation (0 means the component was not useful). `auc_*_only` is the test AUC obtained by flagging the most uncertain edges as anomalies using that uncertainty alone.

In [ ]:
unc = results[results.method == "Uncertainty"]
if len(unc):
    diag_cols = [c for c in unc.columns if c.startswith("lambda_") or (c.startswith("auc_") and c.endswith("_only"))]
    display(unc.groupby(["dataset", "anomaly"])[diag_cols].mean().round(3))

In [ ]:
import matplotlib.pyplot as plt

# Grouped bars: test AUC per (dataset, anomaly type); palette slots 1-2 (validated pair).
COLORS = {"Baseline": "#2a78d6", "Uncertainty": "#eb6834"}
ABBREV = {"temporal-structural-contextual": "t-s-c", "structural-contextual": "s-c",
          "temporal-contextual": "t-c", "temporal": "t", "contextual": "c"}

auc = results.groupby(["dataset", "anomaly", "method"])["auc"].agg(["mean", "std"]).reset_index()
groups = auc[["dataset", "anomaly"]].drop_duplicates().values.tolist()
methods = [m for m in COLORS if m in set(auc.method)]
x = np.arange(len(groups))
width = 0.38

fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(groups)), 4))
for i, method in enumerate(methods):
    sub = auc[auc.method == method].set_index(["dataset", "anomaly"])
    means = np.array([100 * sub.loc[tuple(g), "mean"] if tuple(g) in sub.index else np.nan for g in groups])
    stds = np.array([100 * sub.loc[tuple(g), "std"] if tuple(g) in sub.index else np.nan for g in groups])
    offset = (i - (len(methods) - 1) / 2) * (width + 0.02)
    bars = ax.bar(x + offset, means, width, yerr=np.nan_to_num(stds), color=COLORS[method], label=method,
                  error_kw=dict(ecolor="#555555", lw=1, capsize=2))
    ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8, color="#333333")

ax.set_xticks(x, [f"{d}\n{ABBREV.get(a, a)}" for d, a in groups])
ax.set_ylabel("Test AUC (x100)")
lowest = np.nanmin(100 * auc["mean"].values)
ax.set_ylim(max(0, lowest - 10), 100)
ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.set_title("GraphMixer: baseline vs. uncertainty-aware scoring", loc="left")
plt.tight_layout()
plt.savefig(f"{WORK_DIR}/auc_comparison.png", dpi=150)
plt.show()

## 8. Save outputs
Results, logs and configs are zipped into `/kaggle/working/results.zip` (checkpoints excluded).

In [ ]:
%cd {WORK_DIR}
!zip -qr results.zip EXPERIMENTS/saved_results EXPERIMENTS/logs config_baseline.yaml config_uncertainty.yaml summary.csv auc_comparison.png
!ls -lh results.zip

---
### Reproducing the paper without the uncertainty module
* In this notebook: set `RUN_UNCERTAINTY = False`, and set `NUM_EPOCHS = 50`, `NUM_RUNS = 3`, `NUM_ANOM_SETS = 3`, `LEARNING_RATES = [1e-3, 3e-4, 1e-4, 3e-5, 1e-5, 3e-6, 1e-6]`.
* From the command line: `python train_tgb_linkanomdet.py --config_path=experiments/experiment.yaml`, or any config without an `uncertainty:` section.
* Ablations: change `UNCERTAINTY["components"]`, e.g. `["aleatoric", "epistemic"]` for purely model-based uncertainty.